# M3 GGUF export (Kaggle T4)
Thin runner: all logic lives in `src/training/export_gguf.py`. Merges the v1 adapter and writes GGUFs for local evaluation (`docs/plans/milestone-3.md`, Phase 1). Settings: GPU T4 x2 (one is used), Internet ON, Secret `HF_TOKEN` (write).
Outputs go to the **private scratch** repo `HLeon/diff-mind-v1-gguf`, not the M4 publish repo. Run cells in order.

In [ ]:
# 1. Clone an exact commit (it must be pushed, and must contain src/training/export_gguf.py).
COMMIT = "PASTE_PUSHED_SHA"
assert COMMIT != "PASTE_PUSHED_SHA", "paste the pushed commit SHA"
!git clone https://github.com/hlong24082k/Diff-Mind
%cd Diff-Mind
!git checkout $COMMIT
!git log --oneline -1

In [ ]:
# NOTE: /kaggle/working has only ~12 GB free, so exports go to /tmp (Unsloth also redirects its merged weights there). Hub is the output.
# 2. Install. Same pins as notebook 02 cell 2 (the versions that trained v1).
!pip install -q unsloth==2026.10.11 unsloth_zoo==2026.10.11 trl==1.15.0 peft==0.20.0 transformers==5.16.1 bitsandbytes==0.50.2 accelerate==1.14.0
!pip install -q pyyaml pydantic

In [ ]:
# 3. Secret + single GPU. Must run BEFORE anything imports torch (restart the kernel if you already did).
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# 4. Step 125 (Hub root = best eval_loss = the v1 headline): Q4_K_M for the eval, f16 for the quantization ablation.
!python -m src.training.export_gguf --adapter HLeon/diff-mind-qwen2.5-coder-3b-lora --out /tmp/gguf --name v1-s125 --quant q4_k_m f16 --push-to HLeon/diff-mind-v1-gguf

In [ ]:
# 5. Step 201 (ablation, overfit checkpoint): Q4_K_M only. --expect-step fails fast if last-checkpoint is not step 201.
!python -m src.training.export_gguf --adapter HLeon/diff-mind-qwen2.5-coder-3b-lora --subfolder last-checkpoint --expect-step 201 --out /tmp/gguf --name v1-s201 --quant q4_k_m --push-to HLeon/diff-mind-v1-gguf

In [ ]:
# 6. Verify what landed on the Hub. Expect v1-s125-q4_k_m (~1.9-2.1 GB), v1-s125-f16 (~6 GB), v1-s201-q4_k_m (~1.9-2.1 GB).
from huggingface_hub import HfApi

for f in HfApi().model_info("HLeon/diff-mind-v1-gguf", files_metadata=True).siblings:
    print(f"{(f.size or 0) / 2**30:6.2f} GiB  {f.rfilename}")